In [1]:
import folium
import sqlalchemy 
import pandas as pd
import os
import sqlite3
import re
import unicodedata

from folium import FeatureGroup
from folium.plugins import HeatMap, MarkerCluster
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import create_engine, Column, Integer, String, Enum, Float
from folium import IFrame
from branca.colormap import linear

# Combine two SQL databases together according to rank

In [2]:
# Define the directory for the databases
data_directory = "../Data/"

# Create engines for both databases with the directory paths
engine_main = create_engine(f"sqlite:///{data_directory}main.db", echo=False)
engine_places = create_engine(f"sqlite:///{data_directory}places.db", echo=False)

# Function to merge pokemon tables based on rank with different table names
def merge_pokemon_tables(main_table_name, places_table_name, engine_main, engine_places):
    # Load the tables from both databases into Pandas
    df_main = pd.read_sql(f"SELECT * FROM {main_table_name}", engine_main)
    df_places = pd.read_sql(f"SELECT * FROM {places_table_name}", engine_places)
    
    # Determine the smaller table size
    min_rows = min(len(df_main), len(df_places))
    
    # Sort by 'rank' and keep only the min_rows (to match the smaller table)
    df_main = df_main.sort_values(by="ranking").head(min_rows)
    df_places = df_places.sort_values(by="ranking").head(min_rows)
    
    # Perform a SQL-style JOIN on 'rank'
    merged_df = pd.merge(df_main, df_places, on="ranking", suffixes=("_main", "_places"))
    
    return merged_df

# Merge the ice_pokemon tables (with different names in main.db and places.db)
merged_ice_pokemon = merge_pokemon_tables("ice_pokemon", "coldest_places", engine_main, engine_places)

# Merge the fire_pokemon tables (with different names in main.db and places.db)
merged_fire_pokemon = merge_pokemon_tables("fire_pokemon", "hottest_places", engine_main, engine_places)

# Merge the water_pokemon tables (with different names in main.db and places.db)
merged_water_pokemon = merge_pokemon_tables("water_pokemon", "wettest_places", engine_main, engine_places)

# Save merged tables into a NEW database inside the same directory
engine_merged = create_engine(f"sqlite:///{data_directory}merged.db", echo=False)

# Save the merged ice_pokemon, fire_pokemon, and water_pokemon tables to the merged database
merged_ice_pokemon.to_sql("ice_pokemon", engine_merged, if_exists="replace", index=False)
merged_fire_pokemon.to_sql("fire_pokemon", engine_merged, if_exists="replace", index=False)
merged_water_pokemon.to_sql("water_pokemon", engine_merged, if_exists="replace", index=False)

print("✅ Merged tables saved to Data/merged.db")


✅ Merged tables saved to Data/merged.db


In [3]:
conn = sqlite3.connect('../Data/merged.db')


# Query to get data from all three tables
fire_query = "SELECT * FROM fire_pokemon"
water_query = "SELECT * FROM water_pokemon"
ice_query = "SELECT * FROM ice_pokemon"

# Load the data into separate pandas DataFrames
fire_df = pd.read_sql_query(fire_query, conn)
water_df = pd.read_sql_query(water_query, conn)
ice_df = pd.read_sql_query(ice_query, conn)

In [4]:
# Define the grammar fix function
def fix_grammar_in_dataframe(df):
    # Function to fix the grammar of a single Pokémon description
    def fix_grammar(description):
        # Fix "POKéMON" to "Pokémon"
        description = re.sub(r'POKéMON', 'Pokémon', description)
        
        # Normalize Unicode characters (e.g., handle special characters)
        description = unicodedata.normalize("NFKC", description)
        
        # Replace double spaces with a single space
        description = re.sub(r'\s{2,}', ' ', description)
        
        # Fix broken word "con­tinuously" (which might have hidden characters)
        description = re.sub(r'con­tinuously', 'continuously', description)
        
        # Merge words with a hyphen, such as "X- Y" to "X-Y"
        description = re.sub(r'(\w)- (\w)', r'\1-\2', description)
        
        # Capitalize all uppercase words
        description = re.sub(r'\b([A-Z]+)\b', lambda match: match.group(0).capitalize(), description)
        
        # Ensure a space between words if there is a form feed character
        description = re.sub(r'(\w)\f(\w)', r'\1 \2', description)
        
        # Add space after full stops if missing
        description = re.sub(r'\.(\S)', r'. \1', description)
        
        # Ensure space after full stop and before form feed character
        description = re.sub(r'\.(\f)', r'. \1', description)
        
        # Remove unwanted characters (non-alphanumeric, non-whitespace)
        description = re.sub(r'[^\w\s.,;?!\'"-]', '', description)
        
        # Fix broken word "Un able" to "Unable"
        description = re.sub(r'\bUn\s+able\b', 'Unable', description)
        
        # Remove non-printable characters
        description = ''.join(char for char in description if char.isprintable())
        
        # Fix "Its" followed by certain words to "It's"
        description = re.sub(r'\bIts\b(\s+(highly|feeling|apparently))', r"It's\1", description)
        
        # Keep "Its" if followed by a noun (indicating possession)
        description = re.sub(r'\bIts\b(\s+[a-zA-Z]+)', lambda m: m.group(0), description)
        
        # Fix common mistakes like "doesnt" to "doesn't" and "cant" to "can't"
        description = re.sub(r"\bdoesnt\b", "doesn't", description)
        description = re.sub(r"\bcant\b", "can't", description)
        
        # Standardize "UFO" to uppercase
        description = re.sub(r'\bufo\b', "UFO", description, flags=re.IGNORECASE)
        
        # Ensure possessive "Trainer's" is used correctly
        if "Trainer's" not in description:
            description = re.sub(r'\bTrainers\b', "Trainer's", description)
        
        # Ensure possessive "Pokémon's" is used correctly
        if "Pokémon's" not in description:
            description = re.sub(r'\bPokémons\b', "Pokémon's", description)
        
        # Fix possessive "Skeledirges" to "Skeledirge's"
        description = re.sub(r'\bSkeledirges\b', "Skeledirge's", description)
        
        # Fix "Pokémonanything" to "Pokémon - anything"
        description = re.sub(r'Pokémonanything', 'Pokémon - anything', description)
        
        # Fix "gather ing" to "gathering"
        description = re.sub(r' gather ing', ' gathering', description)
        
        return description

    # Apply the grammar fix function to the "pokemon_description" column
    df['pokemon_description'] = df['pokemon_description'].apply(fix_grammar)
    
    return df


In [5]:
# Apply the function to clean the descriptions
ice_df = fix_grammar_in_dataframe(ice_df)
fire_df = fix_grammar_in_dataframe(fire_df)
water_df = fix_grammar_in_dataframe(water_df)

# Now save the updated DataFrame back to the SQL database
ice_df.to_sql('ice_pokemon', conn, if_exists='replace', index=False)

water_df.to_sql('water_pokemon', conn, if_exists='replace', index=False)

fire_df.to_sql('fire_pokemon', conn, if_exists='replace', index=False)

# Close the connection
conn.close()

# Creating a map

In [6]:
# Define database path
DATABASE_URL = "sqlite:///../Data/merged.db"

# Connect to the database
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)
session = Session()

# Define ORM base
Base = declarative_base()

# Define ORM model for Fire Pokémon
class FirePokemon(Base):
    __tablename__ = "fire_pokemon"
    pokemon_id = Column(Integer, primary_key=True)
    name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    pokemon_description = Column(String)
    pokemon_portrait = Column(String)
    total_stat = Column(Float)
    ranking = Column(Float)

# Define ORM model for Ice Pokémon
class IcePokemon(Base):
    __tablename__ = "ice_pokemon"
    pokemon_id = Column(Integer, primary_key=True)
    name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    pokemon_description = Column(String)
    pokemon_portrait = Column(String)
    total_stat = Column(Float)
    ranking = Column(Float)

# Define ORM model for Water Pokémon
class WaterPokemon(Base):
    __tablename__ = "water_pokemon"
    pokemon_id = Column(Integer, primary_key=True)
    name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    pokemon_description = Column(String)
    pokemon_portrait = Column(String)
    total_stat = Column(Float)
    ranking = Column(Float)

# Fetch Pokémon data
fire_pokemon_entries = session.query(FirePokemon).all()
ice_pokemon_entries = session.query(IcePokemon).all()
water_pokemon_entries = session.query(WaterPokemon).all()  # Fetch Water Pokémon entries

# Initialize the map
map_center = [20.0, 0.0]  # World center
pokemon_map = folium.Map(location=map_center, zoom_start=2)

# Create MarkerCluster groups for Fire, Ice, and Water Pokémon
fire_cluster = MarkerCluster(name="Fire Pokémon").add_to(pokemon_map)
ice_cluster = MarkerCluster(name="Ice Pokémon").add_to(pokemon_map)
water_cluster = MarkerCluster(name="Water Pokémon").add_to(pokemon_map)  # Add Water Pokémon cluster

def add_pokemon_markers(pokemon_entries, cluster_group):
    for pokemon in pokemon_entries:
        lat, lon = pokemon.latitude, pokemon.longitude  

        # Ensure values exist
        name = pokemon.name if pokemon.name else "Unknown"
        description = pokemon.pokemon_description if pokemon.pokemon_description else "No description available"
        stats = pokemon.total_stat if pokemon.total_stat else "N/A"
        ranking = pokemon.ranking if pokemon.ranking else "Unranked"
        portrait_url = pokemon.pokemon_portrait if pokemon.pokemon_portrait else "https://via.placeholder.com/150"
    
        tooltip_html = f"""
        <div style="text-align: center; width: 220px; max-width: 220px; padding: 10px; background-color: white; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); word-wrap: break-word; overflow-wrap: break-word;">
            <h4 style="font-size: 16px; font-weight: bold; color: #D34B29; margin: 0;">{name}</h4>
            <img src="{portrait_url}" width="150px" style="cursor: pointer; border-radius: 8px; max-width: 100%; height: auto;">
            <br><br>
            <b style="font-size: 14px;">Stats:</b> <span style="font-size: 12px; color: #555;">{stats}</span><br>
            <b style="font-size: 14px;">Ranking:</b> <span style="font-size: 12px; color: #555;">{ranking}</span><br>
            <p style="font-size: 12px; text-align: justify; color: #333; margin-top: 8px; line-height: 1.5; word-wrap: break-word; overflow-wrap: break-word; white-space: normal; max-width: 180px; padding: 0; margin: 0;">{description}</p>
        </div>
        """

        tooltip = folium.Tooltip(tooltip_html, sticky=True)  # sticky=True keeps the tooltip visible on hover

        # Create a DivIcon with the Pokémon image
        icon_html = f"""
        <div style="
            background: url('{portrait_url}') no-repeat center center;
            background-size: contain;
            width: 50px; height: 50px;">
        </div>
        """
        div_icon = folium.DivIcon(html=icon_html)

        # Create a marker with the Pokémon image and tooltip
        marker = folium.Marker(
            location=[lat, lon],
            tooltip=tooltip,  # Use Tooltip instead of Popup
            icon=div_icon  # Use Pokémon image as the marker icon
        )

        # Add marker to the cluster group
        marker.add_to(cluster_group)

# Add Fire Pokémon markers
add_pokemon_markers(fire_pokemon_entries, fire_cluster)

# Add Ice Pokémon markers
add_pokemon_markers(ice_pokemon_entries, ice_cluster)

# Add Water Pokémon markers
add_pokemon_markers(water_pokemon_entries, water_cluster) 

# Add groups to the map
pokemon_map.add_child(fire_cluster)
pokemon_map.add_child(ice_cluster)
pokemon_map.add_child(water_cluster) 

/tmp/ipykernel_20004/2521975080.py:10: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


# Getting hottest, coldest and wettest places


In [7]:
# Define file paths for cold, hot, and wet places
filename_cold = '/files/ds105a-2024-project-error_105/Temperature exploration/coldest_places.csv'
filename_hot = '/files/ds105a-2024-project-error_105/Temperature exploration/hottest_places.csv'
filename_wet = '/files/ds105a-2024-project-error_105/Temperature exploration/wettest_places.csv'

# Read CSV files
df_coldest_places = pd.read_csv(filename_cold)
df_hottest_places = pd.read_csv(filename_hot)
df_wettest_places = pd.read_csv(filename_wet)

hotcold_marker_toggle = 0  # Toggle to control the display of hot and cold markers

if hotcold_marker_toggle == 1:
    # Create color scales for hot, cold, and wet places
    hot_colormap = linear.Reds_09.scale(df_hottest_places['temperature'].min(), df_hottest_places['temperature'].max())
    cold_colormap = linear.Blues_09.scale(df_coldest_places['temperature'].min(), df_coldest_places['temperature'].max())
    wet_colormap = linear.Greens_09.scale(df_wettest_places['rainfall'].min(), df_wettest_places['rainfall'].max())

    # Create feature groups for toggleable layers
    hot_layer = folium.FeatureGroup(name="Hottest Places")
    cold_layer = folium.FeatureGroup(name="Coldest Places")
    wet_layer = folium.FeatureGroup(name="Wettest Places")  # New feature group for wettest places

    # Add markers for hottest places
    for idx, row in df_hottest_places.iterrows():
        rank = idx + 1
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
            icon=folium.Icon(color='red', icon='cloud'),
            tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
        ).add_to(hot_layer)

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=10,
            color=hot_colormap(row['temperature']),
            fill=True,
            fill_opacity=0.8,
            tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
        ).add_to(hot_layer)

    # Add markers for coldest places
    for idx, row in df_coldest_places.iterrows():
        rank = idx + 1
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
            icon=folium.Icon(color='blue', icon='cloud'),
            tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
        ).add_to(cold_layer)

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=10,
            color=cold_colormap(row['temperature']),
            fill=True,
            fill_opacity=0.8,
            tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
        ).add_to(cold_layer)

    # Add markers for wettest places
    for idx, row in df_wettest_places.iterrows():
        rank = idx + 1
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Rank: {rank}<br>Region: {row['region']}<br>Rainfall: {row['rainfall']}mm",
            icon=folium.Icon(color='green', icon='tint'),
            tooltip=f"Rank {rank}: {row['region']} ({row['rainfall']}mm)"
        ).add_to(wet_layer)

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=10,
            color=wet_colormap(row['rainfall']),
            fill=True,
            fill_opacity=0.8,
            tooltip=f"Rank {rank}: {row['region']} ({row['rainfall']}mm)"
        ).add_to(wet_layer)

    # Add feature groups to the map
    pokemon_map.add_child(hot_layer)
    pokemon_map.add_child(cold_layer)
    pokemon_map.add_child(wet_layer)  

    # Add legends
    hot_colormap.caption = 'Temperature Scale (Hottest Places)'
    cold_colormap.caption = 'Temperature Scale (Coldest Places)'
    wet_colormap.caption = 'Rainfall Scale (Wettest Places)'  
    hot_colormap.add_to(pokemon_map)
    cold_colormap.add_to(pokemon_map)
    wet_colormap.add_to(pokemon_map)  

# HeatMap for hottest, coldest and wettest places


In [8]:
# Create a color scale for temperatures
colormap_temp = linear.RdYlBu_11.scale(df_coldest_places['temperature'].min(), df_hottest_places['temperature'].max())
# Create a color scale for rainfall (wettest places)
colormap_rain = linear.Blues_09.scale(df_wettest_places['max_rainfall'].min(), df_wettest_places['max_rainfall'].max())

# Function to map temperature/rainfall to color intensity
def get_marker_color(value, min_value, max_value, color_scale):
    # Map value to a hex color
    hex_color = color_scale(value)
    return hex_color

# Heatmap layer for the map
heatmap_layer = folium.FeatureGroup(name="HeatMap")

# Add markers for hottest places (Temperature-based)
for _, row in df_hottest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap_temp),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap_temp),
        fill_opacity=0.8
    ).add_to(heatmap_layer)

# Add markers for coldest places (Temperature-based)
for _, row in df_coldest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap_temp),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap_temp),
        fill_opacity=0.8
    ).add_to(heatmap_layer)

# Add markers for wettest places (Rainfall-based)
for _, row in df_wettest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        popup=f"Region: {row['region']}<br>Rainfall: {row['max_rainfall']}mm",
        color=get_marker_color(row['max_rainfall'], df_wettest_places['max_rainfall'].min(), 
                               df_wettest_places['max_rainfall'].max(), colormap_rain),
        fill=True,
        fill_color=get_marker_color(row['max_rainfall'], df_wettest_places['max_rainfall'].min(), 
                                    df_wettest_places['max_rainfall'].max(), colormap_rain),
        fill_opacity=0.8
    ).add_to(heatmap_layer)

# Add color scale legends to the map
colormap_temp.caption = 'Temperature Scale (°C)'
colormap_rain.caption = 'Rainfall Scale (mm)'  # Caption for the wet places
colormap_temp.add_to(pokemon_map)
colormap_rain.add_to(pokemon_map)

# Add HeatMap for temperature intensity and rainfall intensity
heat_data_temp = [[row['latitude'], row['longitude'], row['temperature']] 
                  for _, row in pd.concat([df_hottest_places, df_coldest_places]).iterrows()]
HeatMap(heat_data_temp).add_to(heatmap_layer)

heat_data_rain = [[row['latitude'], row['longitude'], row['max_rainfall']] 
                  for _, row in df_wettest_places.iterrows()]
HeatMap(heat_data_rain).add_to(heatmap_layer)

# Add the heatmap layer to the map
pokemon_map.add_child(heatmap_layer)


# Saving File


In [9]:
folium.LayerControl().add_to(pokemon_map)

# Save the map to an HTML file
pokemon_map.save("pokemon_map.html")

print("✅ Map generated: pokemon_map.html")


session.close()

✅ Map generated: pokemon_map.html
